# Perception DQN training
#### Payload type: CPU image processing + GPU MLP + Gym simulation + memory/memmap

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import torch
import cv2
import torch.nn as nn
import torch.optim as optim
from torchrl.data import TensorDictReplayBuffer
from torchrl.data import LazyMemmapStorage
from tensordict import TensorDict
import mediapy as media
import gymnasium as gym
from tqdm import tqdm


In [ ]:
# Env wrapper that restricts CarRacing to 3 driving actions (forward / left / right).
# This is done to speed up convergence.
class ActionRestrictedWrapper(gym.ActionWrapper):
	ENV_ACTIONS = (3, 1, 2)

	def __init__(self, env, min_speed_to_steer=0.02):
		super().__init__(env)
		self.action_space = gym.spaces.Discrete(3)
		self.min_speed_to_steer = min_speed_to_steer

	def action(self, action):
		env_action = self.ENV_ACTIONS[int(action)]
		car = getattr(self.env.unwrapped, 'car', None)
		if car is not None and env_action != 3:
			speed = float(np.linalg.norm(car.hull.linearVelocity))
			if speed < self.min_speed_to_steer:
				return 3
		return env_action

In [ ]:
# Global variables
num_episodes = 50
max_steps_per_episode = 1000

# DQN parameters
gamma = 0.99
epsilon = 0.5
epsilon_decay = 0.9999
epsilon_min = 0.01
batch_size = 128
lr = 1e-4
update_target_interval = 1000
replay_buffer_size = 1_000_000

In [ ]:
## Solution
NUM_FORWARD_RAYS = 6
NUM_SIDE_RAYS = 2
RAY_FOV = 1.4
STATE_DIM = NUM_FORWARD_RAYS + NUM_SIDE_RAYS + 3  # rays + speed, steering, gyro


def _is_grass(pixel):
	h, s, v = cv2.cvtColor(pixel.reshape(1, 1, 3).astype(np.uint8), cv2.COLOR_RGB2HSV)[0, 0]
	return 35 <= h <= 90 and s >= 35 and v >= 30


def _is_car(pixel):
	r, g, b = int(pixel[0]), int(pixel[1]), int(pixel[2])
	return r > 120 and r > g + 40 and r > b + 40


def _car_center(track):
	"""Find the car blob nearest the bottom-center of the track view."""
	h, w = track.shape[:2]
	rgb = track.astype(np.int16)
	r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]
	car = ((r > 120) & (r > g + 40) & (r > b + 40)).astype(np.uint8)
	n, _, stats, centroids = cv2.connectedComponentsWithStats(car, connectivity=8)
	if n <= 1:
		return w // 2, h - max(5, h // 12)
	target = np.array([w / 2, h - max(5, h // 12)])
	best, best_dist = None, float('inf')
	for i in range(1, n):
		if stats[i, cv2.CC_STAT_AREA] < 8:
			continue
		cx, cy = centroids[i]
		dist = float(np.linalg.norm(np.array([cx, cy]) - target))
		if dist < best_dist:
			best_dist, best = dist, (int(round(cx)), int(round(cy)))
	return best if best is not None else (w // 2, h - max(5, h // 12))


def _cast_ray(track, cx, cy, angle):
	th, tw = track.shape[:2]
	dx, dy = np.sin(angle), -np.cos(angle)
	for step in range(1, th):
		x, y = int(cx + step * dx), int(cy + step * dy)
		if x < 0 or x >= tw or y < 0 or y >= th:
			return step / th, (x, y)
		px = track[y, x]
		if _is_car(px):
			continue
		if _is_grass(px):
			return step / th, (x, y)
	return 1.0, (int(cx + th * dx), int(cy + th * dy))


def _hud_layout(h, w):
	slot, bar = w / 40.0, h / 40.0
	y_hud = int(round(h - 5 * bar))
	y_bar_top = int(round(h - 4 * bar))
	y_bar_bot = int(round(h - 2 * bar))
	return {
		'y_hud': y_hud,
		'hud_rows': h - y_hud,
		'speed': (round(5 * slot), y_hud, round(6 * slot), h - 1),
		'steering': (round(20 * slot), y_bar_top, round(30 * slot), y_bar_bot),
		'gyro': (round(30 * slot), y_bar_top, min(w - 1, round(40 * slot)), y_bar_bot),
	}


def _hud_masks(image, layout):
	h, w = image.shape[:2]
	masks = {}
	sx1, sy1, sx2, sy2 = layout['speed']
	stx1, sty1, stx2, sty2 = layout['steering']
	gx1, gy1, gx2, gy2 = layout['gyro']

	speed = np.zeros((h, w), dtype=bool)
	speed[sy1:sy2 + 1, sx1:sx2 + 1] = image[sy1:sy2 + 1, sx1:sx2 + 1].min(axis=2) >= 140
	masks['speed'] = speed

	steer = np.zeros((h, w), dtype=bool)
	roi = image[sty1:sty2 + 1, stx1:stx2 + 1]
	green = roi[:, :, 1].astype(np.int16) - np.maximum(roi[:, :, 0], roi[:, :, 2]).astype(np.int16)
	steer[sty1:sty2 + 1, stx1:stx2 + 1] = green > 30
	masks['steering'] = steer

	gyro = np.zeros((h, w), dtype=bool)
	roi = image[gy1:gy2 + 1, gx1:gx2 + 1]
	red = roi[:, :, 0].astype(np.int16) - np.maximum(roi[:, :, 1], roi[:, :, 2]).astype(np.int16)
	gyro[gy1:gy2 + 1, gx1:gx2 + 1] = red > 30
	masks['gyro'] = gyro
	return masks


def _set_pixel(img, x, y, color):
	h, w = img.shape[:2]
	if 0 <= x < w and 0 <= y < h:
		img[y, x] = color


def _draw_line(img, x0, y0, x1, y1, color):
	x0, y0, x1, y1 = int(x0), int(y0), int(x1), int(y1)
	dx, dy = abs(x1 - x0), abs(y1 - y0)
	x, y = x0, y0
	sx = 1 if x0 <= x1 else -1
	sy = 1 if y0 <= y1 else -1
	err = dx - dy
	while True:
		_set_pixel(img, x, y, color)
		if x == x1 and y == y1:
			break
		e2 = 2 * err
		if e2 > -dy:
			err -= dy
			x += sx
		if e2 < dx:
			err += dx
			y += sy


def _border_mask(img, mask, color):
	h, w = mask.shape
	for y in range(h):
		for x in range(w):
			if not mask[y, x]:
				continue
			for dy, dx in ((0, 1), (0, -1), (1, 0), (-1, 0)):
				ny, nx = y + dy, x + dx
				if 0 <= ny < h and 0 <= nx < w and not mask[ny, nx]:
					img[ny, nx] = color


def encode_perception(image):
	if isinstance(image, torch.Tensor):
		image = image.cpu().numpy()
	image = np.clip(image, 0, 255).astype(np.uint8)

	h, w = image.shape[:2]
	layout = _hud_layout(h, w)
	track = image[:layout['y_hud']]
	cx, cy = _car_center(track)

	rays, endpoints = [], []
	for angle in np.linspace(-RAY_FOV / 2, RAY_FOV / 2, NUM_FORWARD_RAYS):
		d, end = _cast_ray(track, cx, cy, angle)
		rays.append(d)
		endpoints.append(end)
	for angle in (-np.pi / 2, np.pi / 2):
		d, end = _cast_ray(track, cx, cy, angle)
		rays.append(d)
		endpoints.append(end)

	masks = _hud_masks(image, layout)
	speed = float(masks['speed'].any(axis=1).sum()) / max(1, layout['hud_rows'])
	if masks['steering'].any():
		xs = np.where(masks['steering'].any(axis=0))[0]
		steering = float(xs.mean() / max(1, w - 1) - 0.5) * 2
	else:
		steering = 0.0
	gyro = float(masks['gyro'].any(axis=1).sum()) / max(1, layout['hud_rows'])

	state = np.array(rays + [speed, np.clip(steering, -1, 1), gyro], dtype=np.float32)
	return state, {
		'track': track, 'ego': (cx, cy), 'endpoints': endpoints,
		'hud_masks': masks, 'hud_rois': layout,
	}


def render_perception_overlay(image, show_rays=True):
	"""Render the perception overlay."""
	if isinstance(image, torch.Tensor):
		image = image.cpu().numpy()
	vis = np.clip(image, 0, 255).astype(np.uint8).copy()
	_, dbg = encode_perception(image)

	if show_rays:
		cx, cy = dbg['ego']
		for x, y in dbg['endpoints']:
			_draw_line(vis, cx, cy, x, y, (255, 0, 255))
			_set_pixel(vis, x, y, (255, 255, 0))
		_set_pixel(vis, cx, cy, (0, 255, 255))

	for name, color in (('speed', (0, 255, 255)), ('steering', (255, 255, 0)), ('gyro', (255, 255, 0))):
		mask = dbg['hud_masks'][name]
		if mask.any():
			_border_mask(vis, mask, color)
	return vis


class CarRacingStateEncoder:
	def __init__(self, device=None):
		self.state_dim = STATE_DIM
		self.device = device or torch.device('cpu')

	def encode(self, image, return_debug=False):
		state, dbg = encode_perception(image)
		t = torch.tensor(state, dtype=torch.float32, device=self.device)
		return (t, dbg) if return_debug else t

	def render_debug(self, image, show_rays=True):
		return render_perception_overlay(image, show_rays=show_rays)



In [ ]:
# DQN agent that encodes pixel observations via CarRacingStateEncoder
class PerceptionDQNAgent:
	"""Small MLP DQN on handcrafted state features."""

	def __init__(self, observation_space, action_space):
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		self.encoder = CarRacingStateEncoder(device=self.device)
		self.state_dim = self.encoder.state_dim
		self.observation_space = observation_space
		self.action_space = action_space
		self.q_network = self._net().to(self.device)
		self.target_network = self._net().to(self.device)
		self.target_network.load_state_dict(self.q_network.state_dict())
		self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
		self.loss_fn = nn.MSELoss()
		self.batch_size = batch_size
		self.buffer = TensorDictReplayBuffer(batch_size=self.batch_size, storage=LazyMemmapStorage(replay_buffer_size))
		self.gamma = gamma
		self.epsilon = epsilon
		self.epsilon_decay = epsilon_decay
		self.epsilon_min = epsilon_min
		self.update_target_interval = update_target_interval
		self.steps_since_last_target_update = 0

	def _net(self):
		return nn.Sequential(
			nn.LayerNorm(self.state_dim),
			nn.Linear(self.state_dim, 64), nn.ReLU(),
			nn.Linear(64, self.action_space.n),
		)

	def _features(self, pixels):
		if isinstance(pixels, torch.Tensor):
			pixels = pixels.cpu().numpy()
		if pixels.ndim == 4:
			return torch.stack([self.encoder.encode(f) for f in pixels])
		return self.encoder.encode(pixels)

	def act(self, state, explore=False):
		if explore and torch.rand(1) < self.epsilon:
			return self.action_space.sample()
		with torch.no_grad():
			q = self.q_network(self._features(state).unsqueeze(0))
			return int(torch.argmax(q, dim=1).item())

	def store_experience(self, td):
		td = td.clone()
		td['state_features'] = self._features(td['pixels']).cpu()
		td['next_state_features'] = self._features(td['next_pixels']).cpu()
		self.buffer.add(td)

	def update(self):
		if len(self.buffer) < self.batch_size:
			return
		b = self.buffer.sample(self.batch_size)
		s, ns = b['state_features'].float().to(self.device), b['next_state_features'].float().to(self.device)
		a = b['action'].long().to(self.device)
		r = b['reward'].to(self.device)
		d = b['done'].to(self.device)
		q = self.q_network(s).gather(1, a.unsqueeze(1))
		with torch.no_grad():
			tgt = r + self.gamma * self.target_network(ns).max(1)[0] * (1 - d)
		loss = self.loss_fn(q, tgt.unsqueeze(1))
		self.optimizer.zero_grad()
		loss.backward()
		self.optimizer.step()
		self.steps_since_last_target_update += 1
		if self.steps_since_last_target_update >= self.update_target_interval:
			self.target_network.load_state_dict(self.q_network.state_dict())
			self.steps_since_last_target_update = 0
		if self.epsilon > self.epsilon_min:
			self.epsilon *= self.epsilon_decay


In [ ]:
# Train perception DQN
env = ActionRestrictedWrapper(gym.make('CarRacing-v3', continuous=False))
perception_dqn_agent = PerceptionDQNAgent(env.observation_space, env.action_space)
episode_rewards_perception_dqn = []

for episode in range(num_episodes):
	state = env.reset()[0]
	episode_reward = 0

	pbar = tqdm(range(max_steps_per_episode), leave=True)
	for step in pbar:
		pbar.set_description(
			f"Episode {episode + 1:>3}/{num_episodes} - "
			f"Reward: {episode_reward:>10.2f}, Epsilon: {perception_dqn_agent.epsilon:<.4f}"
		)
		action = perception_dqn_agent.act(state, explore=True)
		next_state, reward, terminal, truncated, _ = env.step(action)
		perception_dqn_agent.store_experience(TensorDict({
			'pixels': torch.tensor(state, dtype=torch.float32),
			'action': torch.tensor(action, dtype=torch.int64),
			'reward': torch.tensor(reward, dtype=torch.float32),
			'next_pixels': torch.tensor(next_state, dtype=torch.float32),
			'done': torch.tensor(float(terminal or truncated)),
		}))
		perception_dqn_agent.update()
		state, episode_reward = next_state, episode_reward + float(reward)
		if terminal or truncated:
			break
	episode_rewards_perception_dqn.append(episode_reward)

